# Training Model batdetect2 with own data
The training data & annotations as well as the python project batdetect2 are taken from a google drive.
## Run Script to keep Colab alive
Google Colab automatically stops running after a while on inactivity on the notebook.To prevent this, press F12, switch to console and paste and run the code below:
```
function KeepColabAlive() {  
  setInterval(function() { console.log("Keeping Colab alive..."); }, 30000);  
}  
KeepColabAlive();
```
It simulates activity to prevent the notebook from disconnecting from the runtime. Keep the browser window open and the tab running Colab in front.

## Installation of required packages
### downgrade python
For the training of batdetect2 python needs to be version 3.10

In [ ]:
# Install Python 3.10
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-distutils -y

# Install pip for Python 3.10
!wget https://bootstrap.pypa.io/get-pip.py
!python3.10 get-pip.py

# Update alternatives to point python3 to python3.10
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1

# Check the Python version
!python3.10 --version

Mount google drive to get access to the training data and python scripts.

In [ ]:
import os
from google.colab import drive
os.makedirs('cnt/drv', exist_ok=True)
drive.mount('cnt/drv')

Mounted at cnt/drv


Now install the requirements for running batdetect2. The path to the requirements.txt file is the path to the root directory of the python project on the google drive. This path has to be adapted to your needs.

In [ ]:
!pip uninstall -y numpy
!pip install -r '/content/cnt/drv/MyDrive/bat/bd2/cm_bd2/requirements.txt'
!pip install matplotlib_inline
!pip install IPython
!pip install --upgrade matplotlib matplotlib-inline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 153.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 188.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 163.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 166.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 66.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 890.1/890.1 MB 19.5 MB/s  0:00:19
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 42.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 78.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 34.5 MB/s  0:00:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 177.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 43.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.1/557.1 MB 16.8 MB/s  0:00:18
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 75.5 MB/s  0:00:00
   ━━

## Prepare data set for training

Copy training data to a local folder in the runtime system for better performance.

In [ ]:
%mkdir /ann
%cp /content/cnt/drv/MyDrive/bat/bd2/ann.zip /ann
%cd /ann
!unzip ann.zip
%rm ann.zip
%mkdir /wav
%cp /content/cnt/drv/MyDrive/bat/bd2/wav.zip /wav
%cd /wav
!unzip wav.zip
%rm wav.zip
%mkdir /split

Split test and training data. Here also the path in the %cd command has to be adapted.
I had problems with using environment variables containing " . That's why class input and output names are provided directly.

In [ ]:
%env  path_wav=/wav/
%env  path_ann=/ann/
%env  data_set=GermanBats091
%env  path_out=/split
%env  split=0.20
%env  seed=123478
%env  CLASSES_IN="Myotis mystacinus;Myotis brandtii;Plecotus auritus;Plecotus austriacus;Pipistrellus kuhlii;Pipistrellus nathusii"
%env  CLASSES_OUT="Mbart;Mbart;Plecotus;Plecotus;Pipistrellus nathusii;Pipistrellus nathusii"
%cd /content/cnt/drv/MyDrive/bat/bd2/cm_bd2/bat_detect/finetune
!python3.10 prep_data_finetune.py $data_set $path_wav $path_ann $path_out --percent_val $split --rand_seed $seed --input_class_names "Myotis mystacinus;Myotis brandtii;Plecotus auritus;Plecotus austriacus;Pipistrellus kuhlii;Pipistrellus nathusii" --output_class_names "Mbart;Mbart;Plecotus;Plecotus;Pipistrellus nathusii;Pipistrellus nathusii"

## Training
Perform the training of the model.

In [ ]:
# file name of json file of training set
%env TRAIN_DATA=/split/GermanBats091_TRAIN.json
# file name of json file of test set
%env TEST_DATA=/split/GermanBats091_TEST.json
# path to store the result data file
%env OUT_PATH=/content/cnt/drv/MyDrive/bat/bd2/model/
# path to the pretrained model
%env TRAINED_MODEL=../../models/Net2DFast_UK_same.pth.tar
# number of epochs
%env EPOCHS=200
%env O_SCRATCH=--train_from_scratch
#set flag to train only last layer
#%env O_LAST_L=""
# set O_LAST_L=--finetune_only_last_layer
!python3.10 finetune_model.py $path_wav $TRAIN_DATA $TEST_DATA $TRAINED_MODEL --op_model_name ${data_set}.tar --num_epochs $EPOCHS $O_SCRATCH
